# Insurance Premium Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `premium`

## 2 · Project Overview

We predict **insurance premiums** based on policyholder demographics (age, gender, BMI), lifestyle (smoking, exercise), health history (chronic conditions, claims), dependents, region, and policy tier.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a policyholder's age, BMI, smoking status, exercise habits, health history, dependents, region, and policy type, predict their annual insurance premium.

## 5 · Why This Project Matters

- **Premium pricing** is the core of insurance business models.
- Accurate risk assessment prevents adverse selection.
- Understanding premium drivers enables consumer transparency.
- Demonstrates regression with strong multiplicative risk factors.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | age, gender, bmi, num_dependents, smoker, region, policy_type, exercise_freq, chronic_conditions, prev_claims |
| **Target** | premium (continuous, USD) |
| **Range** | ~$500 – $60,000 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "premium"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 insurance policy records with policyholder and risk features.

In [ ]:
np.random.seed(SEED)
n = 8000
age = np.random.randint(18, 70, n)
gender = np.random.choice(["Male", "Female"], n, p=[0.52, 0.48])
bmi = np.round(np.random.normal(27, 5, n).clip(15, 50), 1)
num_dependents = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.2, 0.25, 0.25, 0.15, 0.1, 0.05])
smoker = np.random.choice(["Yes", "No"], n, p=[0.2, 0.8])
region = np.random.choice(["Northeast", "Northwest", "Southeast", "Southwest"], n)
policy_type = np.random.choice(["Basic", "Standard", "Premium"], n, p=[0.35, 0.4, 0.25])
policy_mult = {"Basic": 0.7, "Standard": 1.0, "Premium": 1.5}
pol_val = np.array([policy_mult[p] for p in policy_type])

exercise_freq = np.random.choice(["Never", "Occasional", "Regular", "Daily"], n,
                                  p=[0.2, 0.3, 0.3, 0.2])
exercise_disc = {"Never": 1.1, "Occasional": 1.0, "Regular": 0.92, "Daily": 0.85}
ex_val = np.array([exercise_disc[e] for e in exercise_freq])

chronic_conditions = np.random.poisson(0.5, n).clip(0, 5)
prev_claims = np.random.poisson(1, n).clip(0, 8)

# Premium model
smoker_mult = np.where(smoker == "Yes", 2.5, 1.0)
age_factor = 100 + 15 * age
bmi_factor = np.where(bmi > 30, 200 * (bmi - 30), 0)

premium = (age_factor + bmi_factor + 500 * num_dependents
           + 800 * chronic_conditions + 300 * prev_claims)
premium = premium * smoker_mult * pol_val * ex_val
premium = np.round(premium + np.random.normal(0, 500, n), 2).clip(500, 60000)

df = pd.DataFrame({
    "age": age, "gender": gender, "bmi": bmi, "num_dependents": num_dependents,
    "smoker": smoker, "region": region, "policy_type": policy_type,
    "exercise_freq": exercise_freq, "chronic_conditions": chronic_conditions,
    "prev_claims": prev_claims, "premium": premium
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['premium'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["age"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Age"); axes[0][0].set_ylabel("Premium")
axes[0][0].set_title("Age vs Premium")

axes[0][1].scatter(df["bmi"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("BMI"); axes[0][1].set_ylabel("Premium")
axes[0][1].set_title("BMI vs Premium")

df.boxplot(column=TARGET, by="smoker", ax=axes[0][2])
axes[0][2].set_title("Premium by Smoker Status")

df.boxplot(column=TARGET, by="policy_type", ax=axes[1][0])
axes[1][0].set_title("Premium by Policy Type")

df.boxplot(column=TARGET, by="exercise_freq", ax=axes[1][1])
axes[1][1].set_title("Premium by Exercise")
axes[1][1].tick_params(axis="x", rotation=45)

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `premium`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Premium ($)")

df.boxplot(column=TARGET, by="smoker", ax=axes[1])
axes[1].set_title("Premium: Smoker vs Non-Smoker")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")
print(f"Smoker mean: ${df[df['smoker']=='Yes'][TARGET].mean():,.0f}")
print(f"Non-smoker mean: ${df[df['smoker']=='No'][TARGET].mean():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `gender`, `smoker`, `region`, `policy_type`, `exercise_freq` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Smoker premiums create bimodal distribution — genuine signal.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["bmi_over_30"] = np.maximum(X_train["bmi"] - 30, 0)
X_test["bmi_over_30"] = np.maximum(X_test["bmi"] - 30, 0)

X_train["age_x_chronic"] = X_train["age"] * X_train["chronic_conditions"]
X_test["age_x_chronic"] = X_test["age"] * X_test["chronic_conditions"]

X_train["total_risk_factors"] = X_train["chronic_conditions"] + X_train["prev_claims"]
X_test["total_risk_factors"] = X_test["chronic_conditions"] + X_test["prev_claims"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Smoking status** is the single strongest premium driver (2.5× multiplier).
- **Age** adds ~$15/year linearly.
- **BMI above 30** triggers excess obesity premium.
- **Policy type** (Premium: 1.5× vs Basic: 0.7×) sets coverage tier.
- **Exercise frequency** provides a discount (Daily: 15% off).
- **Chronic conditions** add $800 each.

**Business takeaway:** Smoking cessation programs and exercise incentives are the most cost-effective ways to reduce premiums.

## 26 · Limitations

1. Synthetic data with simplified actuarial models.
2. No medical underwriting details.
3. Missing occupation and income data.
4. Simplified region effects.
5. No longitudinal claims history.

## 27 · How to Improve This Project

1. Use real anonymized insurance claims data.
2. Add detailed medical history and prescription data.
3. Include occupation risk categories.
4. Model policy renewal and lapse rates.
5. Add telematics or wearable device data.

## 28 · Production Considerations

- Deploy for real-time premium quoting.
- Integrate with underwriting workflows.
- Monitor for regulatory compliance (non-discriminatory pricing).
- Update annually with new claims data.
- Output risk scores alongside premiums.

## 29 · Common Mistakes

1. Treating BMI as linearly related to premium (threshold at 30 matters).
2. Not separating smoker/non-smoker populations.
3. Ignoring policy type in pricing models.
4. Using simple averages instead of risk-adjusted pricing.
5. Not accounting for claim history.

## 30 · Mini Challenge / Exercises

1. Remove `smoker` — how much does R² drop?
2. Plot premium vs. age, colored by smoker status.
3. Create `bmi_category` (underweight/normal/overweight/obese) and compare.
4. Build separate models for smoker vs. non-smoker.
5. Calculate the actuarial fair premium vs. actual premium by group.

## 31 · Final Summary / Key Takeaways

1. **Smoking** is the dominant premium multiplier (2.5×).
2. **Age** increases premiums linearly.
3. **BMI above 30** triggers excess charges.
4. **Policy type** sets the coverage and cost tier.
5. **Exercise** provides measurable premium discounts.
6. **Multiplicative risk factors** require tree models or feature engineering.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))